<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 4. Mejorar la Generalización en modelos</font></h1>

<h1><font color="#113D68" size=4>5. Programación de la tasa de aprendizaje</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Programación de la tasa de aprendizaje](#section1)
    * [1.1. Adaptar la tasa de aprendizaje](#section11)
    * [1.2. Ejemplo de línea base](#section12)
* [2. Planificador Lineal](#section2)
* [3. Planificador Exponencial](#section3)
* [4. Programaciones Personalizadas de Tasa de Aprendizaje](#section4)
* [5. Consejos para Utilizar Programaciones de Tasa de Aprendizaje](#section5)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

Entrenar una red neuronal o un modelo grande de aprendizaje profundo es una tarea de optimización difícil. El algoritmo clásico para entrenar redes neuronales se llama **descenso de gradiente estocástico**. Se ha establecido bien que se puede lograr un aumento en el rendimiento y un entrenamiento más rápido en algunos problemas al usar una tasa de aprendizaje que cambia durante el entrenamiento.

En esta sesión aprenderemos qué es una **tasa de aprendizaje programada (_learning rate schedule_)** y cómo puedes usar diferentes programaciones de tasas de aprendizaje para tus modelos de redes neuronales en PyTorch.

- El rol de la programación de la tasa de aprendizaje en el entrenamiento de modelos.
- Cómo usar la programación de la tasa de aprendizaje en el bucle de entrenamiento de PyTorch.
- Cómo configurar tu propia programación de tasa de aprendizaje.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
[PyTorch tutorials.](https://pytorch.org/tutorials/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Documentation: [_How to adjust learning rate_](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Programación de la tasa de aprendizaje</font>

- Descenso de gradiente:
  - Algoritmo de optimización numérica

  - Fórmula: $w := w - \alpha \frac{{dy}}{{dw}}$

    - w: parámetro (ej. peso en red neuronal)

    - y: objetivo (ej. función de pérdida)

    - $\alpha$: tasa de aprendizaje

- Tasa de aprendizaje ($\alpha$):
  - Controla el tamaño del paso en la actualización de parámetros

- Enfoques para la tasa de aprendizaje:
  1. Tasa constante

  2. Programación de tasa de aprendizaje (adaptativa)

- Ventajas de la tasa adaptativa:
  - Mejora el rendimiento

  - Reduce el tiempo de entrenamiento

- Actualización de la tasa de aprendizaje:
  - Generalmente una vez por época

  - Posible actualización en cada paso, pero menos común

- Razones para actualizar por época:
  - Permite evaluar el rendimiento de la red

  - Facilita la determinación de cómo ajustar la tasa

- Importancia de la programación de tasa de aprendizaje:
  - Adapta el aprendizaje a diferentes etapas del entrenamiento
  
  - Puede mejorar la convergencia y el rendimiento final del modelo

<a id="section11"></a>
# <font color="#004D7F" size=5>1.1. Adaptar la tasa de aprendizaje</font>

- Inicio del entrenamiento:
  - Preferencia por tasa de aprendizaje mayor

  - Mejora rápida de la red

  - Aceleración del progreso

- En modelos complejos:
  - Posibilidad de aumentar gradualmente la tasa al inicio

  - Permite explorar diferentes dimensiones de predicción

- Final del entrenamiento:
  - Necesidad de tasa de aprendizaje menor

  - Evita sobreajuste cuando el rendimiento es cercano al óptimo

- Técnicas comunes:
  - Reducción de la tasa de aprendizaje con el tiempo

  - Beneficios:

    - Grandes cambios al principio

    - Actualizaciones más pequeñas al final

- Ventajas de este enfoque:
  - Aprendizaje rápido de buenos pesos inicialmente

  - Afinamiento preciso en etapas posteriores

  - Balance entre exploración inicial y refinamiento final

  - Adaptación a las necesidades específicas del modelo y datos

  - Mejora la eficiencia del entrenamiento

  - Puede llevar a mejor convergencia y rendimiento final

<a id="section12"></a>
# <font color="#004D7F" size=5>1.2. Ejemplo de línea base</font>

- Ejemplo con modelo de línea base
  - Problema: Clasificación binaria de la ionosfera

  - Nombre del archivo: `ionosphere.csv`

  - Valores de entrada: pequeños valores numéricos

  - Misma escala para todos los valores
  
  - Tamaño manejable

  - Datos pre-procesados y normalizados

- Estructura del modelo de red neuronal:
  - Una única capa oculta

  - 34 neuronas en la capa oculta

  - Función de activación: ReLU

- Capa de salida:
  - Una sola neurona

  - Función de activación: Sigmoide

  - Genera valores similares a probabilidades


In [1]:
# Descarga de datos (necesario en Colab): se guardan en ./Datasets
import os, urllib.request

os.makedirs('Datasets', exist_ok=True)
urls = {
    'ionosphere.csv':
        'https://raw.githubusercontent.com/jbrownlee/Datasets/master/ionosphere.csv',
}
for nombre, url in urls.items():
    if not os.path.exists(f'Datasets/{nombre}'):
        urllib.request.urlretrieve(url, f'Datasets/{nombre}')

In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

dataframe = pd.read_csv("Datasets/ionosphere.csv", header=None)
dataset = dataframe.values
X = dataset[:,0:34].astype(float)
y = dataset[:,34]

encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True)

model = nn.Sequential(
    nn.Linear(34, 34),
    nn.ReLU(),
    nn.Linear(34, 1),
    nn.Sigmoid()
)

n_epochs = 50
batch_size = 24
batch_start = torch.arange(0, len(X_train), batch_size)
lr = 0.1
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)
model.train()
for epoch in range(n_epochs):
    for start in batch_start:
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print("Época %d: SGD lr=%.4f" % (epoch, optimizer.param_groups[0]["lr"]))

model.eval()
y_pred = model(X_test)
acc = (y_pred.round() == y_test).float().mean()
acc = float(acc)
print("Model accuracy: %.2f%%" % (acc*100))

Época 0: SGD lr=0.1000
Época 1: SGD lr=0.1000
Época 2: SGD lr=0.1000
Época 3: SGD lr=0.1000
Época 4: SGD lr=0.1000
Época 5: SGD lr=0.1000
Época 6: SGD lr=0.1000
Época 7: SGD lr=0.1000
Época 8: SGD lr=0.1000
Época 9: SGD lr=0.1000
Época 10: SGD lr=0.1000
Época 11: SGD lr=0.1000
Época 12: SGD lr=0.1000
Época 13: SGD lr=0.1000


Época 14: SGD lr=0.1000
Época 15: SGD lr=0.1000
Época 16: SGD lr=0.1000
Época 17: SGD lr=0.1000
Época 18: SGD lr=0.1000
Época 19: SGD lr=0.1000
Época 20: SGD lr=0.1000
Época 21: SGD lr=0.1000
Época 22: SGD lr=0.1000
Época 23: SGD lr=0.1000
Época 24: SGD lr=0.1000
Época 25: SGD lr=0.1000
Época 26: SGD lr=0.1000
Época 27: SGD lr=0.1000
Época 28: SGD lr=0.1000


Época 29: SGD lr=0.1000
Época 30: SGD lr=0.1000
Época 31: SGD lr=0.1000
Época 32: SGD lr=0.1000
Época 33: SGD lr=0.1000
Época 34: SGD lr=0.1000
Época 35: SGD lr=0.1000
Época 36: SGD lr=0.1000
Época 37: SGD lr=0.1000
Época 38: SGD lr=0.1000
Época 39: SGD lr=0.1000
Época 40: SGD lr=0.1000
Época 41: SGD lr=0.1000
Época 42: SGD lr=0.1000
Época 43: SGD lr=0.1000
Época 44: SGD lr=0.1000
Época 45: SGD lr=0.1000
Época 46: SGD lr=0.1000
Época 47: SGD lr=0.1000
Época 48: SGD lr=0.1000
Época 49: SGD lr=0.1000
Model accuracy: 88.68%


Puedes confirmar que la tasa de aprendizaje no cambió durante todo el proceso de entrenamiento. Ahora, hagamos que el proceso de entrenamiento comience con una tasa de aprendizaje mayor y termine con una tasa menor.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Planificador Lineal</font>

- Programación de tasa de aprendizaje en PyTorch:
  - Submódulo: `torch.optim.lr_scheduler`

  - Requiere el optimizador como primer argumento

  - Puede necesitar argumentos adicionales según el programador

- Configuración del ejemplo:

  - Algoritmo: Descenso de gradiente estocástico (SGD)

  - Tasa de aprendizaje inicial: 0.1 (fija)

  - Duración del entrenamiento: 50 épocas

- Acceso a la tasa de aprendizaje:
  - Ubicación: `optimizer.param_groups["lr"]`
  
  - Tipo de dato: valor flotante

  - Impresión de la tasa de aprendizaje al final de cada época

- Ventajas de este enfoque:
  - Permite monitorear la tasa de aprendizaje durante el entrenamiento

  - Facilita la comprensión del comportamiento del optimizador

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`LinearLR`](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.LinearLR.html)

In [3]:
import torch.optim.lr_scheduler as lr_scheduler

n_epochs = 50
batch_size = 24
batch_start = torch.arange(0, len(X_train), batch_size)
lr = 0.1
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)
scheduler = lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.5, total_iters=30)

- Uso de `LinearLR()`:
  - Tipo: Programador de tasa lineal

  
  - Parámetros adicionales:
  
    - `start_factor`: 1.0
  
    - `end_factor`: 0.5
  
    - `total_iters`: 30

- Funcionamiento de `LinearLR`:
  - Requiere ejecutar `step()` dentro del bucle de entrenamiento

  - Factor multiplicativo disminuye de 1.0 a 0.5 en 30 pasos iguales

  - Después de 30 pasos, el factor se mantiene en 0.5

  - Factor se multiplica por la tasa de aprendizaje original

- Cambio esperado en la tasa de aprendizaje:
  - Inicio: 0.1 × 1.0 = 0.1

  - Final: 0.1 × 0.5 = 0.05

- Ventajas de este enfoque:
  - Permite una reducción gradual y controlada de la tasa de aprendizaje

  - Facilita el ajuste fino del modelo en las últimas etapas del entrenamiento

- Consideraciones:
  - La elección de parámetros puede afectar significativamente el rendimiento

  - Puede requerir experimentación para encontrar los valores óptimos

In [4]:
model.train()
for epoch in range(n_epochs):
    for start in batch_start:
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    before_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()  # actualizar la tasa una vez por época
    after_lr = optimizer.param_groups[0]["lr"]
    print("Época %d: SGD lr %.4f -> %.4f" % (epoch, before_lr, after_lr))

model.eval()
y_pred = model(X_test)
acc = (y_pred.round() == y_test).float().mean()
acc = float(acc)
print("Accuracy del modelo: %.2f%%" % (acc*100))

Época 0: SGD lr 0.1000 -> 0.0983
Época 1: SGD lr 0.0983 -> 0.0967
Época 2: SGD lr 0.0967 -> 0.0950
Época 3: SGD lr 0.0950 -> 0.0933
Época 4: SGD lr 0.0933 -> 0.0917
Época 5: SGD lr 0.0917 -> 0.0900
Época 6: SGD lr 0.0900 -> 0.0883
Época 7: SGD lr 0.0883 -> 0.0867
Época 8: SGD lr 0.0867 -> 0.0850
Época 9: SGD lr 0.0850 -> 0.0833
Época 10: SGD lr 0.0833 -> 0.0817
Época 11: SGD lr 0.0817 -> 0.0800
Época 12: SGD lr 0.0800 -> 0.0783
Época 13: SGD lr 0.0783 -> 0.0767
Época 14: SGD lr 0.0767 -> 0.0750
Época 15: SGD lr 0.0750 -> 0.0733
Época 16: SGD lr 0.0733 -> 0.0717
Época 17: SGD lr 0.0717 -> 0.0700
Época 18: SGD lr 0.0700 -> 0.0683
Época 19: SGD lr 0.0683 -> 0.0667
Época 20: SGD lr 0.0667 -> 0.0650
Época 21: SGD lr 0.0650 -> 0.0633
Época 22: SGD lr 0.0633 -> 0.0617
Época 23: SGD lr 0.0617 -> 0.0600
Época 24: SGD lr 0.0600 -> 0.0583
Época 25: SGD lr 0.0583 -> 0.0567
Época 26: SGD lr 0.0567 -> 0.0550
Época 27: SGD lr 0.0550 -> 0.0533


Época 28: SGD lr 0.0533 -> 0.0517
Época 29: SGD lr 0.0517 -> 0.0500
Época 30: SGD lr 0.0500 -> 0.0500
Época 31: SGD lr 0.0500 -> 0.0500
Época 32: SGD lr 0.0500 -> 0.0500
Época 33: SGD lr 0.0500 -> 0.0500
Época 34: SGD lr 0.0500 -> 0.0500
Época 35: SGD lr 0.0500 -> 0.0500
Época 36: SGD lr 0.0500 -> 0.0500
Época 37: SGD lr 0.0500 -> 0.0500
Época 38: SGD lr 0.0500 -> 0.0500
Época 39: SGD lr 0.0500 -> 0.0500
Época 40: SGD lr 0.0500 -> 0.0500
Época 41: SGD lr 0.0500 -> 0.0500
Época 42: SGD lr 0.0500 -> 0.0500
Época 43: SGD lr 0.0500 -> 0.0500
Época 44: SGD lr 0.0500 -> 0.0500
Época 45: SGD lr 0.0500 -> 0.0500
Época 46: SGD lr 0.0500 -> 0.0500
Época 47: SGD lr 0.0500 -> 0.0500
Época 48: SGD lr 0.0500 -> 0.0500
Época 49: SGD lr 0.0500 -> 0.0500
Accuracy del modelo: 94.34%


Lo primero que notarás es la mejora en el accuracy, que es la razón por la cual estarías interesado en utilizar una programación de tasa de aprendizaje diferente.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Planificador Exponencial</font>

Además de `LinearLR()`, también puedes usar `ExponentialLR()`. 
En este caso, la tasa de aprendizaje se actualiza multiplicando por un factor constante `gamma` en cada actualización del programador. Su sintaxis es la siguiente:

- Funcionamiento:
  - Actualiza la tasa de aprendizaje multiplicando por un factor constante (gamma)

  - Se aplica en cada actualización del programador

- Parámetros clave:
  - `optimizer`: el optimizador a actualizar

  - `gamma`: factor de decaimiento (menor que 1 para reducir la tasa)

- Comportamiento:
  - Disminución exponencial de la tasa de aprendizaje

  - Decaimiento más rápido al principio, más lento después

- Ejemplo de actualización:
  - Tasa inicial: 0.1

  - Después de 1 paso: 0.1 * 0.99 = 0.099

  - Después de 2 pasos: 0.099 * 0.99 = 0.09801

  - Y así sucesivamente

- Ventajas:
  - Permite un decaimiento suave y continuo de la tasa de aprendizaje

  - Útil para ajuste fino gradual del modelo

- Consideraciones:
  - La elección de gamma es crucial para el rendimiento

  - Puede requerir experimentación para encontrar el valor óptimo

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre [`ExponentialLR`](https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ExponentialLR.html)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
PyTorch Documentation: [_How to adjust learning rate_](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)

In [5]:
n_epochs = 50
batch_size = 24
batch_start = torch.arange(0, len(X_train), batch_size)
lr = 0.1
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)
scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=0.99)  # lr_n = lr_0 * gamma^n
model.train()
for epoch in range(n_epochs):
    for start in batch_start:
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    before_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()
    after_lr = optimizer.param_groups[0]["lr"]
    print("Época %d: SGD lr %.4f -> %.4f" % (epoch, before_lr, after_lr))

model.eval()
y_pred = model(X_test)
acc = (y_pred.round() == y_test).float().mean()
acc = float(acc)
print("Accuracy modelo: %.2f%%" % (acc*100))

Época 0: SGD lr 0.1000 -> 0.0990
Época 1: SGD lr 0.0990 -> 0.0980
Época 2: SGD lr 0.0980 -> 0.0970
Época 3: SGD lr 0.0970 -> 0.0961
Época 4: SGD lr 0.0961 -> 0.0951
Época 5: SGD lr 0.0951 -> 0.0941
Época 6: SGD lr 0.0941 -> 0.0932
Época 7: SGD lr 0.0932 -> 0.0923
Época 8: SGD lr 0.0923 -> 0.0914
Época 9: SGD lr 0.0914 -> 0.0904
Época 10: SGD lr 0.0904 -> 0.0895
Época 11: SGD lr 0.0895 -> 0.0886
Época 12: SGD lr 0.0886 -> 0.0878
Época 13: SGD lr 0.0878 -> 0.0869
Época 14: SGD lr 0.0869 -> 0.0860
Época 15: SGD lr 0.0860 -> 0.0851
Época 16: SGD lr 0.0851 -> 0.0843
Época 17: SGD lr 0.0843 -> 0.0835
Época 18: SGD lr 0.0835 -> 0.0826
Época 19: SGD lr 0.0826 -> 0.0818
Época 20: SGD lr 0.0818 -> 0.0810
Época 21: SGD lr 0.0810 -> 0.0802
Época 22: SGD lr 0.0802 -> 0.0794


Época 23: SGD lr 0.0794 -> 0.0786
Época 24: SGD lr 0.0786 -> 0.0778
Época 25: SGD lr 0.0778 -> 0.0770
Época 26: SGD lr 0.0770 -> 0.0762
Época 27: SGD lr 0.0762 -> 0.0755
Época 28: SGD lr 0.0755 -> 0.0747
Época 29: SGD lr 0.0747 -> 0.0740
Época 30: SGD lr 0.0740 -> 0.0732
Época 31: SGD lr 0.0732 -> 0.0725
Época 32: SGD lr 0.0725 -> 0.0718
Época 33: SGD lr 0.0718 -> 0.0711
Época 34: SGD lr 0.0711 -> 0.0703
Época 35: SGD lr 0.0703 -> 0.0696
Época 36: SGD lr 0.0696 -> 0.0689
Época 37: SGD lr 0.0689 -> 0.0683
Época 38: SGD lr 0.0683 -> 0.0676
Época 39: SGD lr 0.0676 -> 0.0669
Época 40: SGD lr 0.0669 -> 0.0662
Época 41: SGD lr 0.0662 -> 0.0656
Época 42: SGD lr 0.0656 -> 0.0649
Época 43: SGD lr 0.0649 -> 0.0643
Época 44: SGD lr 0.0643 -> 0.0636
Época 45: SGD lr 0.0636 -> 0.0630
Época 46: SGD lr 0.0630 -> 0.0624


Época 47: SGD lr 0.0624 -> 0.0617
Época 48: SGD lr 0.0617 -> 0.0611
Época 49: SGD lr 0.0611 -> 0.0605
Accuracy modelo: 95.28%


---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section4"></a>
# <font color="#004D7F" size=6>4. Programación Personalizada en la Tasa de Aprendizaje</font>

No existe una regla general que diga que una programación de tasa de aprendizaje en particular funcione mejor para todos los casos. A veces, es posible que desees tener una programación especial de la tasa de aprendizaje que PyTorch no proporcione. En tales casos, puedes definir una programación de tasa de aprendizaje personalizada utilizando una función propia.

Por ejemplo, supongamos que quieres una tasa de aprendizaje que siga la fórmula:

$$
    lr_n = \frac{lr_0}{1 + \alpha n}
$$

donde $lr_0$ es la tasa de aprendizaje inicial en la época 0 y $\alpha$ es una constante. Puedes implementar una función que, dado el número de época $n$\), calcule la tasa de aprendizaje $lr_n$.

Esta aproximación permite una mayor personalización en el proceso de entrenamiento, adaptándose a las características únicas de cada problema o modelo. Útil cuando las opciones predefinidas de PyTorch no son suficientes

- Ejemplo de fórmula personalizada:
  - $lr_n = \frac{lr_0}{1 + \alpha n}$

    - _lr_n_: tasa de aprendizaje en la época _n_

    - _lr_0_: tasa de aprendizaje inicial (época 0)

    - $\alpha$: constante de ajuste

    - _n_: número de época

- Ventajas de programaciones personalizadas:
  - Flexibilidad para adaptar la tasa de aprendizaje a necesidades específicas

  - Control preciso sobre el comportamiento del decaimiento

- Consideraciones:
  - Requiere entendimiento profundo del problema y del comportamiento deseado

  - Puede necesitar experimentación para encontrar los parámetros óptimos

- Aplicaciones:
  - Problemas específicos donde las programaciones estándar no son efectivas

  - Investigación de nuevas estrategias de optimización

In [6]:
# LambdaLR multiplica el lr inicial por el valor que devuelve esta función
def lr_lambda(epoch):
    alpha = 0.1
    return 1. / (1. + alpha * epoch)

Luego, puedes configurar un `LambdaLR()` para actualizar la tasa de aprendizaje según esta función personalizada:

In [7]:
n_epochs = 50
batch_size = 24
batch_start = torch.arange(0, len(X_train), batch_size)
lr = 0.1
loss_fn = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)
scheduler = lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
model.train()

Sequential(
  (0): Linear(in_features=34, out_features=34, bias=True)
  (1): ReLU()
  (2): Linear(in_features=34, out_features=1, bias=True)
  (3): Sigmoid()
)

Ahora podemos ejecutar el entrenamiento.

In [8]:
for epoch in range(n_epochs):
    for start in batch_start:
        X_batch = X_train[start:start+batch_size]
        y_batch = y_train[start:start+batch_size]
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    before_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()
    after_lr = optimizer.param_groups[0]["lr"]
    print("Época %d: SGD lr %.4f -> %.4f" % (epoch, before_lr, after_lr))

model.eval()
y_pred = model(X_test)
acc = (y_pred.round() == y_test).float().mean()
acc = float(acc)
print("Accuracy del modelo: %.2f%%" % (acc*100))

Época 0: SGD lr 0.1000 -> 0.0909
Época 1: SGD lr 0.0909 -> 0.0833
Época 2: SGD lr 0.0833 -> 0.0769
Época 3: SGD lr 0.0769 -> 0.0714
Época 4: SGD lr 0.0714 -> 0.0667
Época 5: SGD lr 0.0667 -> 0.0625
Época 6: SGD lr 0.0625 -> 0.0588


Época 7: SGD lr 0.0588 -> 0.0556
Época 8: SGD lr 0.0556 -> 0.0526
Época 9: SGD lr 0.0526 -> 0.0500
Época 10: SGD lr 0.0500 -> 0.0476
Época 11: SGD lr 0.0476 -> 0.0455
Época 12: SGD lr 0.0455 -> 0.0435
Época 13: SGD lr 0.0435 -> 0.0417
Época 14: SGD lr 0.0417 -> 0.0400
Época 15: SGD lr 0.0400 -> 0.0385
Época 16: SGD lr 0.0385 -> 0.0370
Época 17: SGD lr 0.0370 -> 0.0357
Época 18: SGD lr 0.0357 -> 0.0345
Época 19: SGD lr 0.0345 -> 0.0333
Época 20: SGD lr 0.0333 -> 0.0323
Época 21: SGD lr 0.0323 -> 0.0312


Época 22: SGD lr 0.0312 -> 0.0303
Época 23: SGD lr 0.0303 -> 0.0294
Época 24: SGD lr 0.0294 -> 0.0286
Época 25: SGD lr 0.0286 -> 0.0278
Época 26: SGD lr 0.0278 -> 0.0270
Época 27: SGD lr 0.0270 -> 0.0263
Época 28: SGD lr 0.0263 -> 0.0256


Época 29: SGD lr 0.0256 -> 0.0250
Época 30: SGD lr 0.0250 -> 0.0244
Época 31: SGD lr 0.0244 -> 0.0238
Época 32: SGD lr 0.0238 -> 0.0233
Época 33: SGD lr 0.0233 -> 0.0227


Época 34: SGD lr 0.0227 -> 0.0222
Época 35: SGD lr 0.0222 -> 0.0217
Época 36: SGD lr 0.0217 -> 0.0213
Época 37: SGD lr 0.0213 -> 0.0208
Época 38: SGD lr 0.0208 -> 0.0204
Época 39: SGD lr 0.0204 -> 0.0200
Época 40: SGD lr 0.0200 -> 0.0196


Época 41: SGD lr 0.0196 -> 0.0192


Época 42: SGD lr 0.0192 -> 0.0189


Época 43: SGD lr 0.0189 -> 0.0185
Época 44: SGD lr 0.0185 -> 0.0182
Época 45: SGD lr 0.0182 -> 0.0179
Época 46: SGD lr 0.0179 -> 0.0175
Época 47: SGD lr 0.0175 -> 0.0172
Época 48: SGD lr 0.0172 -> 0.0169
Época 49: SGD lr 0.0169 -> 0.0167
Accuracy del modelo: 95.28%


**Nota:** Aunque la función proporcionada a `LambdaLR()` asume un argumento `epoch`, no está vinculada necesariamente a las épocas en el bucle de entrenamiento. En realidad, cuenta cuántas veces invocas `scheduler.step()`.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section5"></a>
# <font color="#004D7F" size=6>5. Consejos para Utilizar Programaciones de Tasa de Aprendizaje</font>

A continuación, se enumeran algunos consejos y trucos a considerar al usar programaciones de tasa de aprendizaje con redes neuronales:

- **Aumenta la tasa de aprendizaje inicial:** Dado que la tasa de aprendizaje probablemente disminuirá con el tiempo, comienza con un valor más grande. 

    - Una tasa de aprendizaje mayor provocará cambios más grandes en los pesos al principio, lo que te permitirá beneficiarte del ajuste fino al final.

- **Usa un momentum alto:** Muchos optimizadores pueden considerar el momentum. 

    - Utilizar un valor alto ayudará al algoritmo de optimización a seguir haciendo actualizaciones en la dirección correcta cuando la tasa de aprendizaje se reduzca a valores pequeños.

- **Experimenta con diferentes programaciones:** No está claro qué programación de tasa de aprendizaje funcionará mejor en cada problema, por lo que es recomendable probar varias configuraciones para ver cuál se ajusta mejor. 

    - También es útil probar programaciones que cambien de manera exponencial e incluso aquellas que respondan a la precisión del modelo en los conjuntos de datos de entrenamiento o prueba.

<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>